# REGRAS DE NEGOCIO - SISTEMA STCP

Sistema de decisao inteligente para otimizacao de carregamento de autocarros electricos.

**8 Regras implementadas:**
1. Limites de Potencia
2. Prioridades de Carregamento
3. Limites de Temperatura
4. Disponibilidade de Estacoes
5. Distancia ao Deposito
6. Tempo Estimado de Carregamento
7. Balanceamento de Carga
8. Saude da Bateria


## REGRA 1: Limites de Potencia

**Objetivo**: Garantir que nenhuma estacao excede a sua potencia maxima.

**Como funciona:**
- Cada estacao tem potencia maxima de **120 kW**
- Bateria de cada autocarro tem capacidade de **350 kWh**
- Niveis criticos: **20%** (minimo), **80%** (ideal)

**Resultado:** Identificar se os limites estao a ser respeitados.

In [0]:
# REGRA 1: Configuracao de Limites
# Nota: usa *_raw para funcionar sem o simulador activo
POTENCIA_MAXIMA_ESTACAO = 120.0  # kW
BATERIA_MAXIMA_AUTOCARRO = 350.0  # kWh
BATERIA_MINIMA_SEGURANCA = 20.0  # %
BATERIA_IDEAL = 80.0  # %

print("="*80)
print("REGRA 1: LIMITES DE POTENCIA")
print("="*80)
print(f"Potencia maxima por estacao: {POTENCIA_MAXIMA_ESTACAO} kW")
print(f"Capacidade bateria: {BATERIA_MAXIMA_AUTOCARRO} kWh")
print(f"Nivel critico: {BATERIA_MINIMA_SEGURANCA}%")
print(f"Nivel ideal: {BATERIA_IDEAL}%")

display(spark.sql(f"""
    SELECT
        device_id,
        ROUND(currentPower, 1) as potencia_atual_kw,
        {POTENCIA_MAXIMA_ESTACAO} as limite_kw,
        CASE
            WHEN currentPower > {POTENCIA_MAXIMA_ESTACAO}       THEN 'EXCEDE LIMITE'
            WHEN currentPower > {POTENCIA_MAXIMA_ESTACAO * 0.9} THEN 'PROXIMO DO LIMITE'
            ELSE 'OK'
        END as status_potencia
    FROM (
        SELECT device_id, currentPower,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    ) WHERE rn = 1
    ORDER BY currentPower DESC
"""))

display(spark.sql(f"""
    SELECT
        device_id,
        ROUND(batteryLevel, 1) as bateria_percent,
        CASE
            WHEN batteryLevel < {BATERIA_MINIMA_SEGURANCA} THEN 'CRITICO'
            WHEN batteryLevel < 50                         THEN 'BAIXO'
            WHEN batteryLevel > {BATERIA_IDEAL}            THEN 'IDEAL'
            ELSE 'OK'
        END as status_bateria
    FROM (
        SELECT device_id, batteryLevel,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    ) WHERE rn = 1
    ORDER BY batteryLevel ASC
"""))

REGRA 1: LIMITES DE POTENCIA
Potencia maxima por estacao: 120.0 kW
Capacidade bateria: 350.0 kWh
Nivel critico: 20.0%
Nivel ideal: 80.0%


device_id,potencia_atual_kw,limite_kw,status_potencia
CS-001,120.0,120.0,PROXIMO DO LIMITE
CS-003,120.0,120.0,PROXIMO DO LIMITE
CS-002,0.0,120.0,OK


device_id,bateria_percent,status_bateria
BUS-001,17.8,CRITICO
BUS-002,45.9,BAIXO
BUS-003,81.2,IDEAL


## REGRA 2: Prioridades de Carregamento

**Objetivo**: Decidir qual autocarro deve carregar primeiro com base em prioridades.

**Como funciona:**
- Cada rota tem uma **prioridade** (Alta=30pts, Normal=20pts)
- Score de urgencia = Pontos_Prioridade + Pontos_Bateria
  - Se bateria < 20% (minimo) -> +60 pontos
  - Se bateria entre 20% e 30% -> +30 pontos
  - Caso contrario -> 0 pontos

**Prioridades definidas:** (distâncias alinhadas com o simulador Python)
- BUS-001 / Linha 200 (8,64 km): Alta
- BUS-002 / Linha 201 (8,39 km): Normal
- BUS-003 / Linha 202 (9,48 km): Normal

**Resultado:** Lista ordenada de autocarros por urgencia.

In [0]:
# REGRA 2: Criar tabela de prioridades
# Distancias alinhadas com o simulador Python (ROUTE_STOPS):
#   BUS-001 / Linha 200 (8.64 km) -> Alta
#   BUS-002 / Linha 201 (8.39 km) -> Normal
#   BUS-003 / Linha 202 (9.48 km) -> Normal
# SOC minimo de seguranca: 20% (70 kWh)
spark.sql("DROP TABLE IF EXISTS route_priorities")

prioridades = spark.createDataFrame([
    ("BUS-001", "200", "Bolhao - Castelo Queijo", "Alta",   8.64, 20.0),
    ("BUS-002", "201", "Aliados - Viso",           "Normal", 8.39, 20.0),
    ("BUS-003", "202", "Aliados - Passeio Alegre", "Normal", 9.48, 20.0)
], ["device_id", "route_id", "route_name", "priority", "distance_km", "min_battery_required"])

prioridades.write.mode("overwrite").saveAsTable("route_priorities")

print("="*80)
print("REGRA 2: PRIORIDADES DE CARREGAMENTO")
print("="*80)

# Nota: usa bus_telemetry_raw para funcionar sem o simulador activo
display(spark.sql("""
    WITH latest_bus AS (
        SELECT device_id, batteryLevel,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    ),
    urgency_calc AS (
        SELECT
            l.device_id,
            ROUND(l.batteryLevel, 1) as bateria_atual,
            r.route_id,
            r.route_name,
            r.priority,
            r.min_battery_required as bateria_minima,
            CASE
                WHEN r.priority = 'Alta'   THEN 30
                WHEN r.priority = 'Normal' THEN 20
                ELSE 10
            END as pontos_prioridade,
            CASE
                WHEN l.batteryLevel < r.min_battery_required        THEN 60
                WHEN l.batteryLevel < (r.min_battery_required + 10) THEN 30
                ELSE 0
            END as pontos_bateria
        FROM latest_bus l
        JOIN route_priorities r ON l.device_id = r.device_id
        WHERE l.rn = 1
    )
    SELECT
        device_id,
        route_id,
        route_name,
        priority as prioridade,
        bateria_atual,
        bateria_minima,
        pontos_prioridade,
        pontos_bateria,
        (pontos_prioridade + pontos_bateria) as score_urgencia,
        CASE
            WHEN (pontos_prioridade + pontos_bateria) >= 70 THEN 'URGENTE'
            WHEN (pontos_prioridade + pontos_bateria) >= 50 THEN 'PRIORITARIO'
            WHEN (pontos_prioridade + pontos_bateria) >= 30 THEN 'NORMAL'
            ELSE 'NAO NECESSARIO'
        END as classificacao
    FROM urgency_calc
    ORDER BY score_urgencia DESC
"""))

REGRA 2: PRIORIDADES DE CARREGAMENTO


device_id,route_id,route_name,prioridade,bateria_atual,bateria_minima,pontos_prioridade,pontos_bateria,score_urgencia,classificacao
BUS-001,200,Bolhao - Castelo Queijo,Alta,17.8,20.0,30,60,90,URGENTE
BUS-002,201,Aliados - Viso,Normal,45.9,20.0,20,0,20,NAO NECESSARIO
BUS-003,202,Aliados - Passeio Alegre,Normal,81.2,20.0,20,0,20,NAO NECESSARIO


## REGRA 3: Limites de Temperatura

**Objetivo**: Proteger baterias e carregadores de sobreaquecimento.

**Como funciona:**
- **Bateria**:
  - Temperatura maxima: **45 graus C** -> PARAR carregamento (potencia = 0 kW)
  - Temperatura > 35 graus C -> Reduzir para 80 kW
  - Temperatura < 15 graus C -> Pre-aquecer (50 kW)
  - Temperatura ideal: **15-35 graus C** -> Potencia normal (120 kW)

- **Carregador**:
  - Temperatura maxima: **60 graus C** -> Aguardar arrefecimento

**Accoes automaticas:**
- T > 45 graus C -> Potencia = 0 kW
- T > 35 graus C -> Potencia = 80 kW
- T < 15 graus C -> Potencia = 50 kW
- 15 <= T <= 35 graus C -> Potencia = 120 kW

In [0]:
# REGRA 3: Limites de Temperatura
# Nota: usa *_raw para funcionar sem o simulador activo
TEMP_BATERIA_MAX = 45.0
TEMP_BATERIA_IDEAL_MIN = 15.0
TEMP_BATERIA_IDEAL_MAX = 35.0
TEMP_CARREGADOR_MAX = 60.0

print("="*80)
print("REGRA 3: LIMITES DE TEMPERATURA")
print("="*80)
print(f"Bateria - Maximo: {TEMP_BATERIA_MAX}C")
print(f"Bateria - Ideal: {TEMP_BATERIA_IDEAL_MIN}C - {TEMP_BATERIA_IDEAL_MAX}C")
print(f"Carregador - Maximo: {TEMP_CARREGADOR_MAX}C")

print("\n--- TEMPERATURA DAS BATERIAS ---")
display(spark.sql(f"""
    SELECT
        device_id,
        ROUND(batteryLevel, 1) as bateria,
        ROUND(batteryTemperature, 1) as temperatura_C,
        CASE
            WHEN batteryTemperature > {TEMP_BATERIA_MAX}       THEN 'CRITICO'
            WHEN batteryTemperature > {TEMP_BATERIA_IDEAL_MAX} THEN 'ALTO'
            WHEN batteryTemperature < {TEMP_BATERIA_IDEAL_MIN} THEN 'FRIO'
            ELSE 'IDEAL'
        END as status_temp,
        CASE
            WHEN batteryTemperature > {TEMP_BATERIA_MAX}       THEN 'PARAR carregamento'
            WHEN batteryTemperature > {TEMP_BATERIA_IDEAL_MAX} THEN 'REDUZIR para 80 kW'
            WHEN batteryTemperature < {TEMP_BATERIA_IDEAL_MIN} THEN 'PRE-AQUECER (50 kW)'
            ELSE 'Potencia normal (120 kW)'
        END as acao_recomendada,
        CASE
            WHEN batteryTemperature > {TEMP_BATERIA_MAX}       THEN 0.0
            WHEN batteryTemperature > {TEMP_BATERIA_IDEAL_MAX} THEN 80.0
            WHEN batteryTemperature < {TEMP_BATERIA_IDEAL_MIN} THEN 50.0
            ELSE 120.0
        END as potencia_ajustada_kw
    FROM (
        SELECT device_id, batteryLevel, batteryTemperature,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    ) WHERE rn = 1
    ORDER BY batteryTemperature DESC
"""))

print("\n--- TEMPERATURA DOS CARREGADORES ---")
display(spark.sql(f"""
    SELECT
        device_id,
        state as estado,
        ROUND(chargerTemperature, 1) as temperatura_C,
        CASE
            WHEN chargerTemperature > {TEMP_CARREGADOR_MAX}       THEN 'SOBREAQUECIMENTO'
            WHEN chargerTemperature > {TEMP_CARREGADOR_MAX * 0.9} THEN 'PROXIMO DO LIMITE'
            ELSE 'NORMAL'
        END as status_temp,
        CASE
            WHEN chargerTemperature > {TEMP_CARREGADOR_MAX}       THEN 'AGUARDAR arrefecimento'
            WHEN chargerTemperature > {TEMP_CARREGADOR_MAX * 0.9} THEN 'MONITORIZAR'
            ELSE 'Operacao normal'
        END as acao
    FROM (
        SELECT device_id, state, chargerTemperature,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    ) WHERE rn = 1
    ORDER BY chargerTemperature DESC
"""))

REGRA 3: LIMITES DE TEMPERATURA
Bateria - Maximo: 45.0C
Bateria - Ideal: 15.0C - 35.0C
Carregador - Maximo: 60.0C

--- TEMPERATURA DAS BATERIAS ---


device_id,bateria,temperatura_C,status_temp,acao_recomendada,potencia_ajustada_kw
BUS-003,81.2,26.3,IDEAL,Potencia normal (120 kW),120.0
BUS-002,45.9,26.0,IDEAL,Potencia normal (120 kW),120.0
BUS-001,17.8,23.2,IDEAL,Potencia normal (120 kW),120.0



--- TEMPERATURA DOS CARREGADORES ---


device_id,estado,temperatura_C,status_temp,acao
CS-001,occupied,45.0,NORMAL,Operacao normal
CS-003,occupied,35.7,NORMAL,Operacao normal
CS-002,fault,28.7,NORMAL,Operacao normal


## REGRA 4: Disponibilidade de Estacoes

**Objetivo**: Verificar quais estacoes estao disponiveis para alocar aos autocarros.

**Como funciona:**
- **available** + Temperatura < 60 graus C -> DISPONIVEL
- **occupied** -> OCUPADO (aguardar)
- **fault** -> EM MANUTENCAO (nao alocar)
- Temperatura >= 60 graus C -> ARREFECIMENTO NECESSARIO

**Estados possiveis (simulador):**
- available (livre)
- occupied (a carregar)
- fault (avaria)

**Resultado:** Lista de estacoes e sua disponibilidade real.

In [0]:
# REGRA 4: Disponibilidade de Estacoes
# Estados reais do simulador: 'available', 'occupied', 'fault'
# Nota: usa charger_telemetry_raw para funcionar sem o simulador activo
print("="*80)
print("REGRA 4: DISPONIBILIDADE DE ESTACOES")
print("="*80)

display(spark.sql("""
    WITH latest_chargers AS (
        SELECT
            device_id,
            state,
            ROUND(currentPower, 1) as potencia_atual,
            connectedBus,
            ROUND(chargerTemperature, 1) as temperatura,
            ROUND(latitude, 6) as lat,
            ROUND(longitude, 6) as lon,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    SELECT
        device_id,
        state as estado,
        potencia_atual,
        connectedBus as autocarro_conectado,
        temperatura,
        CONCAT(lat, ', ', lon) as localizacao,
        CASE
            WHEN state = 'available' AND temperatura < 60.0 THEN 'DISPONIVEL'
            WHEN state = 'occupied'                         THEN 'OCUPADO'
            WHEN temperatura >= 60.0                        THEN 'ARREFECIMENTO'
            WHEN state = 'fault'                            THEN 'MANUTENCAO'
            ELSE 'INDISPONIVEL'
        END as disponibilidade,
        CASE
            WHEN state = 'available' AND temperatura < 60.0 THEN 1
            ELSE 0
        END as pode_alocar
    FROM latest_chargers
    WHERE rn = 1
    ORDER BY pode_alocar DESC, device_id
"""))

print("\n--- RESUMO ---")
display(spark.sql("""
    WITH latest_chargers AS (
        SELECT state, chargerTemperature,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    )
    SELECT
        COUNT(*) as total_estacoes,
        SUM(CASE WHEN state = 'available' AND chargerTemperature < 60.0 THEN 1 ELSE 0 END) as disponiveis,
        SUM(CASE WHEN state = 'occupied'                                 THEN 1 ELSE 0 END) as ocupadas,
        SUM(CASE WHEN chargerTemperature >= 60.0                         THEN 1 ELSE 0 END) as arrefecimento,
        SUM(CASE WHEN state = 'fault'                                    THEN 1 ELSE 0 END) as em_manutencao
    FROM latest_chargers
    WHERE rn = 1
"""))

REGRA 4: DISPONIBILIDADE DE ESTACOES


device_id,estado,potencia_atual,autocarro_conectado,temperatura,localizacao,disponibilidade,pode_alocar
CS-001,occupied,120.0,BUS-002,45.0,"41.152661, -8.579658",OCUPADO,0
CS-002,fault,0.0,null,28.7,"41.152671, -8.579668",MANUTENCAO,0
CS-003,occupied,120.0,BUS-003,35.7,"41.152681, -8.579678",OCUPADO,0



--- RESUMO ---


total_estacoes,disponiveis,ocupadas,arrefecimento,em_manutencao
3,0,2,0,1


## REGRA 5: Distancia ao Deposito

**Objetivo**: Calcular se cada autocarro consegue chegar ao deposito com a bateria actual.

**Como funciona:**
- Localizacao do deposito: **lat=41.152661, lon=-8.579658** (Via Norte / Bonjóia, STCP)
- Consumo medio: **1.2 kWh/km** (CaetanoBus e.City Gold -- consistente com o projecto)
- Margem de seguranca: **20%** (multiplicador 1.2)

**Calculo:**
1. Calcular distancia do autocarro ao deposito (formula Haversine simplificada)
2. Calcular autonomia actual = bateria_kWh / 1.2
3. Comparar: autonomia >= distancia x 1.2?

**Classificacao:**
- CRITICO: Nao chega (urgencia 100)
- ALERTA: Margem reduzida (urgencia 60)
- OK: Consegue chegar (urgencia 0)

In [0]:
# REGRA 5: Distancia ao Deposito
# Consumo: 1.2 kWh/km (CaetanoBus e.City Gold -- consistente com o projecto)
# Nota: usa bus_telemetry_raw para funcionar sem o simulador activo
# Localização: Via Norte / Bonjóia, alinhada com STATION_LOCATION em config.py
# e com as coordenadas de CS-001 no simulador.
DEPOT_LAT = 41.152661
DEPOT_LON = -8.579658
CONSUMO_POR_KM = 1.2
MARGEM_SEGURANCA = 1.2

print("="*80)
print("REGRA 5: DISTANCIA AO DEPOSITO")
print("="*80)
print(f"Via Norte / Bonjoia: ({DEPOT_LAT}, {DEPOT_LON})")
print(f"Consumo: {CONSUMO_POR_KM} kWh/km")
print(f"Margem: {MARGEM_SEGURANCA}x (20%)")

display(spark.sql(f"""
    WITH latest_bus AS (
        SELECT
            device_id,
            ROUND(batteryLevel, 1) as bateria_percent,
            ROUND(batteryKwh, 1) as bateria_kwh,
            latitude,
            longitude,
            state,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    ),
    distance_calc AS (
        SELECT
            device_id,
            bateria_percent,
            bateria_kwh,
            state,
            ROUND(
                111.0 * SQRT(
                    POW(({DEPOT_LAT} - latitude), 2) +
                    POW(({DEPOT_LON} - longitude) * COS(latitude * 3.14159 / 180), 2)
                ),
                2
            ) as distancia_depot_km,
            ROUND(bateria_kwh / {CONSUMO_POR_KM}, 1) as autonomia_km
        FROM latest_bus
        WHERE rn = 1
    )
    SELECT
        device_id,
        bateria_percent,
        distancia_depot_km,
        autonomia_km,
        ROUND(distancia_depot_km * {CONSUMO_POR_KM} * {MARGEM_SEGURANCA}, 1) as energia_necessaria_kwh,
        ROUND(autonomia_km - (distancia_depot_km * {MARGEM_SEGURANCA}), 1) as margem_km,
        CASE
            WHEN autonomia_km < (distancia_depot_km * {MARGEM_SEGURANCA}) THEN 'CRITICO - Nao chega'
            WHEN autonomia_km < (distancia_depot_km * 1.5)                THEN 'ALERTA - Margem reduzida'
            ELSE 'OK - Consegue chegar'
        END as status_alcance,
        CASE
            WHEN autonomia_km < (distancia_depot_km * {MARGEM_SEGURANCA}) THEN 100
            WHEN autonomia_km < (distancia_depot_km * 1.5)                THEN 60
            ELSE 0
        END as urgencia_alcance
    FROM distance_calc
    ORDER BY urgencia_alcance DESC
"""))

REGRA 5: DISTANCIA AO DEPOSITO
Via Norte / Bonjoia: (41.152661, -8.579658)
Consumo: 1.2 kWh/km
Margem: 1.2x (20%)


device_id,bateria_percent,distancia_depot_km,autonomia_km,energia_necessaria_kwh,margem_km,status_alcance,urgencia_alcance
BUS-001,17.8,2.47,51.8,3.6,48.8,OK - Consegue chegar,0
BUS-002,45.9,0.0,133.9,0.0,133.9,OK - Consegue chegar,0
BUS-003,81.2,0.0,236.8,0.0,236.8,OK - Consegue chegar,0


## REGRA 6: Tempo Estimado de Carregamento

**Objetivo**: Calcular quanto tempo cada autocarro precisa para atingir 80% de carga.

**Como funciona:**
- Capacidade bateria: **350 kWh**
- Energia a carregar = (80% x 350) - bateria_actual_kWh
- Tempo = Energia / Potencia de carregamento

**Cenarios de potencia (baseados nas regras de temperatura):**
- **120 kW** -> Temperatura normal (15 a 35 graus C)
- **80 kW** -> Temperatura elevada (> 35 graus C) -- caso actual do simulador
- **50 kW** -> Temperatura baixa (< 15 graus C)

**Classificacao:**
- < 30 min -> Carregamento rapido
- 30-60 min -> Carregamento medio
- > 60 min -> Carregamento longo

In [0]:
# REGRA 6: Tempo Estimado de Carregamento
# Target: 80% (280 kWh) -- definido nas regras do projecto
# Cenarios: 120 kW (normal), 80 kW (T>35C), 50 kW (T<15C)
# Nota: autocarros com bateria >= 80% aparecem no resultado, rotulados 'Ja carregado'
# pelo CASE abaixo (sem filtro WHERE). Usa bus_telemetry_raw para funcionar sem o simulador activo.
print("="*80)
print("REGRA 6: TEMPO ESTIMADO DE CARREGAMENTO")
print("="*80)

display(spark.sql("""
    WITH latest_bus AS (
        SELECT
            device_id,
            ROUND(batteryLevel, 1) as bateria_atual_percent,
            ROUND(batteryKwh, 1) as bateria_atual_kwh,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    ),
    charging_calc AS (
        SELECT
            device_id,
            bateria_atual_percent,
            ROUND((80.0 / 100.0) * 350.0 - bateria_atual_kwh, 1) as energia_a_carregar_kwh
        FROM latest_bus
        WHERE rn = 1
    )
    SELECT
        device_id,
        bateria_atual_percent as bateria_atual,
        80.0 as bateria_objetivo_percent,
        GREATEST(energia_a_carregar_kwh, 0) as energia_a_carregar_kwh,
        CAST(ROUND(GREATEST(energia_a_carregar_kwh, 0) / 120.0 * 60, 0) AS INT) as tempo_120kw_min,
        CAST(ROUND(GREATEST(energia_a_carregar_kwh, 0) /  80.0 * 60, 0) AS INT) as tempo_80kw_min,
        CAST(ROUND(GREATEST(energia_a_carregar_kwh, 0) /  50.0 * 60, 0) AS INT) as tempo_50kw_min,
        CASE
            WHEN bateria_atual_percent >= 80.0              THEN 'Ja carregado'
            WHEN (energia_a_carregar_kwh / 120.0) <= 0.5   THEN 'Rapido (<30min)'
            WHEN (energia_a_carregar_kwh / 120.0) <= 1.0   THEN 'Medio (30-60min)'
            ELSE 'Longo (>60min)'
        END as categoria_tempo
    FROM charging_calc
    ORDER BY energia_a_carregar_kwh DESC
"""))

REGRA 6: TEMPO ESTIMADO DE CARREGAMENTO


device_id,bateria_atual,bateria_objetivo_percent,energia_a_carregar_kwh,tempo_120kw_min,tempo_80kw_min,tempo_50kw_min,categoria_tempo
BUS-001,17.8,80.0,217.9,109,163,261,Longo (>60min)
BUS-002,45.9,80.0,119.3,60,89,143,Medio (30-60min)
BUS-003,81.2,80.0,0.0,0,0,0,Ja carregado


## REGRA 7: Balanceamento de Carga

**Objetivo**: Distribuir carga entre estacoes sem sobrecarregar a rede electrica.

**Como funciona:**
- **Potencia total disponivel**: 360 kW (3 estacoes x 120 kW)
- **Limite da rede**: 360 kW

**Estados da rede:**
- **> 360 kW**: SOBRECARGA -> Reduzir para 80% por estacao
- **324-360 kW** (90-100%): ALERTA -> Reduzir para 90%
- **252-324 kW** (70-90%): CARGA ELEVADA -> Monitorizar
- **< 252 kW** (< 70%): NORMAL -> Potencia maxima

**Accao automatica:** Ajustar potencia de todas as estacoes proporcionalmente.

In [0]:
# REGRA 7: Balanceamento de Carga
# Estados reais do simulador: 'available', 'occupied', 'fault'
# Nota: usa charger_telemetry_raw para funcionar sem o simulador activo
POTENCIA_TOTAL_DISPONIVEL = 360.0
LIMITE_CARGA_REDE = 360.0

print("="*80)
print("REGRA 7: BALANCEAMENTO DE CARGA")
print("="*80)
print(f"Potencia total: {POTENCIA_TOTAL_DISPONIVEL} kW")
print(f"Limite rede: {LIMITE_CARGA_REDE} kW")

display(spark.sql(f"""
    WITH latest_chargers AS (
        SELECT
            device_id,
            currentPower,
            state,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    ),
    carga_total AS (
        SELECT
            ROUND(SUM(currentPower), 1) as carga_atual_kw,
            COUNT(CASE WHEN state = 'occupied'  THEN 1 END) as estacoes_ativas,
            COUNT(CASE WHEN state = 'available' THEN 1 END) as estacoes_disponiveis
        FROM latest_chargers
        WHERE rn = 1
    )
    SELECT
        carga_atual_kw,
        estacoes_ativas,
        estacoes_disponiveis,
        ROUND({POTENCIA_TOTAL_DISPONIVEL} - carga_atual_kw, 1) as potencia_livre_kw,
        ROUND({LIMITE_CARGA_REDE} - carga_atual_kw, 1) as margem_rede_kw,
        ROUND((carga_atual_kw / {LIMITE_CARGA_REDE}) * 100, 1) as percentagem_uso,
        CASE
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE}       THEN 'SOBRECARGA'
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE * 0.9} THEN 'ALERTA'
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE * 0.7} THEN 'CARGA ELEVADA'
            ELSE 'NORMAL'
        END as status_rede,
        CASE
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE}       THEN ROUND((carga_atual_kw / GREATEST(estacoes_ativas, 1)) * 0.8, 1)
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE * 0.9} THEN ROUND((carga_atual_kw / GREATEST(estacoes_ativas, 1)) * 0.9, 1)
            ELSE 120.0
        END as potencia_maxima_ajustada
    FROM carga_total
"""))

print("\n--- CONSUMO POR ESTACAO ---")
display(spark.sql("""
    SELECT
        device_id,
        state,
        ROUND(currentPower, 1) as potencia_kw,
        connectedBus,
        ROUND((currentPower / 120.0) * 100, 1) as percentagem_capacidade
    FROM (
        SELECT device_id, state, currentPower, connectedBus,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    ) WHERE rn = 1
    ORDER BY currentPower DESC
"""))

REGRA 7: BALANCEAMENTO DE CARGA
Potencia total: 360.0 kW
Limite rede: 360.0 kW


carga_atual_kw,estacoes_ativas,estacoes_disponiveis,potencia_livre_kw,margem_rede_kw,percentagem_uso,status_rede,potencia_maxima_ajustada
240.0,2,0,120.0,120.0,66.7,NORMAL,120.0



--- CONSUMO POR ESTACAO ---


device_id,state,potencia_kw,connectedBus,percentagem_capacidade
CS-001,occupied,120.0,BUS-002,100.0
CS-003,occupied,120.0,BUS-003,100.0
CS-002,fault,0.0,null,0.0


## REGRA 8: Saude da Bateria (Longevidade)

**Objetivo**: Preservar vida util da bateria evitando descargas profundas e cargas excessivas.

**Como funciona:**
- **Ciclos de vida**: ~3000 ciclos esperados
- **Faixa ideal**: 20-80% (maximiza longevidade)

**Zonas de operacao:**
- **< 20%**: DESCARGA PROFUNDA (reduz vida util) -> Carregar ate 80%
- **20-80%**: FAIXA IDEAL (boa para longevidade) -> Carregar ate 80%
- **> 80%**: SOBRECARGA (pode degradar) -> NAO carregar mais

**Impacto:**
- Operar sempre 20-80% -> +30% vida util
- Descargas < 20% frequentes -> -20% vida util
- Cargas > 80% frequentes -> -15% vida util

In [0]:
# REGRA 8: Saude da Bateria
# Target de carregamento: 80% em todos os casos (consistente com o projecto)
# Nota: usa bus_telemetry_raw para funcionar sem o simulador activo
CICLOS_MAX = 3000
FAIXA_IDEAL_MIN = 20.0
FAIXA_IDEAL_MAX = 80.0

print("="*80)
print("REGRA 8: SAUDE DA BATERIA")
print("="*80)
print(f"Ciclos esperados: {CICLOS_MAX}")
print(f"Faixa ideal: {FAIXA_IDEAL_MIN}% - {FAIXA_IDEAL_MAX}%")

display(spark.sql(f"""
    WITH latest_bus AS (
        SELECT
            device_id,
            ROUND(batteryLevel, 1) as bateria_percent,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    )
    SELECT
        device_id,
        bateria_percent,
        CASE
            WHEN bateria_percent < {FAIXA_IDEAL_MIN} THEN 'DESCARGA PROFUNDA'
            WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN 'SOBRECARGA'
            ELSE 'FAIXA IDEAL'
        END as zona_operacao,
        CASE
            WHEN bateria_percent < {FAIXA_IDEAL_MIN} THEN 'Reduz vida util (-20%)'
            WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN 'Pode degradar (-15%)'
            ELSE 'Otimo para longevidade (+30%)'
        END as impacto_longevidade,
        {FAIXA_IDEAL_MAX} as nivel_alvo_carregamento,
        CASE
            WHEN bateria_percent < {FAIXA_IDEAL_MIN} THEN 'Carregar ate 80%'
            WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN 'NAO carregar mais'
            ELSE 'Carregar ate 80%'
        END as recomendacao
    FROM latest_bus
    WHERE rn = 1
    ORDER BY bateria_percent ASC
"""))

REGRA 8: SAUDE DA BATERIA
Ciclos esperados: 3000
Faixa ideal: 20.0% - 80.0%


device_id,bateria_percent,zona_operacao,impacto_longevidade,nivel_alvo_carregamento,recomendacao
BUS-001,17.8,DESCARGA PROFUNDA,Reduz vida util (-20%),80.0,Carregar ate 80%
BUS-002,45.9,FAIXA IDEAL,Otimo para longevidade (+30%),80.0,Carregar ate 80%
BUS-003,81.2,SOBRECARGA,Pode degradar (-15%),80.0,NAO carregar mais
